In [3]:
import os
import time
import json
import requests
import numpy as np
import pandas as pd
import streamlit as st
from dotenv import load_dotenv
from tmdbv3api import TMDb, Movie
from requests.adapters import HTTPAdapter, Retry
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()

False

In [4]:
tmdb = TMDb()
tmdb.api_key = st.secrets.get("API_KEY")
tmdb_movie = Movie()

In [5]:
def make_session():
    s = requests.Session()
    retries = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=0.7,  # 0.7, 1.4, 2.8, ...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],  # only retry idempotent GET
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept-Language": "en-US,en;q=0.9",
        }
    )
    return s


HTTP = make_session()

In [6]:
def fetch_html(url, timeout=25):
    resp = HTTP.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.content


def get_wiki_tables(year: int) -> list[pd.DataFrame]:
    url = f"https://en.wikipedia.org/wiki/List_of_American_films_of_{year}"
    html = fetch_html(url)
    tables = pd.read_html(html, header=0)
    wanted = {"Title", "Cast and crew"}
    filtered = [t for t in tables if wanted.issubset(set(t.columns))]
    return filtered or tables

In [7]:
TMDB_QPS = 4  # target queries per second
MAX_WORKERS = 8  # threads for I/O-bound tasks
CACHE_FILE = "tmdb_genres_cache.json"
_GENRE_CACHE = {}


def load_cache():
    if os.path.exists(CACHE_FILE):
        try:
            _GENRE_CACHE.update(json.load(open(CACHE_FILE, "r")))
        except Exception:
            pass


def save_cache():
    try:
        with open(CACHE_FILE, "w") as f:
            json.dump(_GENRE_CACHE, f)
    except Exception:
        pass


load_cache()

In [8]:
def fetch_genre_for_title(title):
    key = (title or "").strip().lower()
    if not key:
        return key, np.nan
    if key in _GENRE_CACHE:
        return key, _GENRE_CACHE[key]

    # Search
    try:
        results = tmdb_movie.search(title)
    except Exception:
        _GENRE_CACHE[key] = np.nan
        return key, np.nan
    if not results:
        _GENRE_CACHE[key] = np.nan
        return key, np.nan

    movie_id = getattr(results[0], "id", None)
    if not movie_id:
        _GENRE_CACHE[key] = np.nan
        return key, np.nan

    # Details with retries via shared Session
    try:
        r = HTTP.get(
            f"https://api.themoviedb.org/3/movie/{movie_id}",
            params={"api_key": tmdb.api_key},
            timeout=20,
        )
        if r.status_code == 429:
            time.sleep(1.5)
            r = HTTP.get(
                f"https://api.themoviedb.org/3/movie/{movie_id}",
                params={"api_key": tmdb.api_key},
                timeout=20,
            )
        r.raise_for_status()
        names = [g.get("name") for g in r.json().get("genres", []) if g.get("name")]
        val = " ".join(names) if names else np.nan
    except requests.RequestException:
        val = np.nan

    _GENRE_CACHE[key] = val
    return key, val

In [9]:
def enrich_genres_vectorized(df_titles: pd.Series) -> pd.Series:
    titles = df_titles.astype(str).str.strip()
    unique_titles = sorted(set(titles.str.lower()))
    to_fetch = [t for t in unique_titles if t and t not in _GENRE_CACHE]

    if to_fetch:
        interval = 1.0 / TMDB_QPS
        last = 0.0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {}
            for t in to_fetch:
                now = time.time()
                if now - last < interval:
                    time.sleep(interval - (now - last))
                last = time.time()
                futures[ex.submit(fetch_genre_for_title, t)] = t

            for fut in as_completed(futures):
                k, v = fut.result()
        save_cache()

    return titles.str.lower().map(_GENRE_CACHE).astype("object")

In [10]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [11]:
def get_actor1(x):
    return (x.split("screenplay); ")[-1]).split(", ")[0]

In [12]:
def get_actor2(x):
    return (
        (x.split("screenplay); ")[-1]).split(", ")[1]
        if len((x.split("screenplay); ")[-1]).split(", ")) > 1
        else "unknown"
    )

In [13]:
def get_actor3(x):
    return (
        (x.split("screenplay); ")[-1]).split(", ")[2]
        if len((x.split("screenplay); ")[-1]).split(", ")) > 2
        else "unknown"
    )

In [14]:
def process_movies(year):
    try:
        tables = get_wiki_tables(year)
        dfs = tables[:4] if len(tables) > 4 else tables
        df = pd.concat(dfs, ignore_index=True)
        df["Title"] = df["Title"].astype(str).str.strip()
        df["Cast and crew"] = df["Cast and crew"].astype(str).str.strip()

        # Vectorized enrichment
        df["genres"] = enrich_genres_vectorized(df["Title"])

        # Rest unchanged
        df = df[["Title", "Cast and crew", "genres"]]
        df["director_name"] = df["Cast and crew"].map(get_director)
        df["actor_1_name"] = df["Cast and crew"].map(get_actor1)
        df["actor_2_name"] = df["Cast and crew"].map(get_actor2)
        df["actor_3_name"] = df["Cast and crew"].map(get_actor3)

        df = df.rename(columns={"Title": "movie_title"})
        df["actor_2_name"] = df["actor_2_name"].fillna("unknown")
        df["actor_3_name"] = df["actor_3_name"].fillna("unknown")
        df["movie_title"] = df["movie_title"].str.lower()
        df["comb"] = (
            df["actor_1_name"]
            + " "
            + df["actor_2_name"]
            + " "
            + df["actor_3_name"]
            + " "
            + df["director_name"]
            + " "
            + df["genres"].fillna("")
        )
        return df[
            [
                "director_name",
                "actor_1_name",
                "actor_2_name",
                "actor_3_name",
                "genres",
                "movie_title",
                "comb",
            ]
        ]
    except Exception as e:
        print(f"Error processing movies for {year}: {e}")
        return pd.DataFrame()

In [15]:
movies_2018 = process_movies(2018)

In [ ]:
movies_2019 = process_movies(2019)

In [ ]:
movies_2021 = process_movies(2021)

In [ ]:
movies_2022 = process_movies(2022)

In [ ]:
movies_2023 = process_movies(2023)

In [ ]:
movies_2024 = process_movies(2024)

In [ ]:
combined_movies = pd.concat(
    [
        movies_2018,
        movies_2019,
        movies_2021,
        movies_2022,
        movies_2023,
        movies_2024,
    ],
    ignore_index=True,
)

In [ ]:
old_df = pd.read_csv("./datasets/new_data.csv")
final_df = pd.concat([old_df, combined_movies], ignore_index=True)

In [ ]:
final_df = final_df.dropna(how="any")

In [ ]:
final_df.to_csv("./datasets/final_data.csv", index=False)

In [ ]:
print(final_df.isna().sum())